In [1]:
!pip install update

In [6]:
!apt-get update
!apt-get install -y portaudio19-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,106 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:14 htt

In [7]:
!pip install PyAudio

  Using cached PyAudio-0.2.14.tar.gz (47 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for PyAudio: filename=pyaudio-0.2.14-cp312-cp312-linux_x86_64.whl size=68670 sha256=644b4d706bb8605e1c675ef147b9bbee1c6e06c34d29453177f5470a6dc8228f
  Stored in directory: /root/.cache/pip/wheels/68/c7/33/c6a6b210cb5819ec5c219928c794a447742a7d86d21c0b92e6
Successfully built PyAudio


In [8]:
!pip install PyAudio==0.2.11

  Using cached PyAudio-0.2.11.tar.gz (37 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for PyAudio: filename=PyAudio-0.2.11-cp312-cp312-linux_x86_64.whl size=52929 sha256=4ea4c563e3f7881423d5c6012c5b5edf759cde3f3fc44115190a51a7caae3b4c
  Stored in directory: /root/.cache/pip/wheels/65/3b/dd/f89390a284b9042070707568809def73ce4f3ecdd116e74fd8
Successfully built PyAudio
  Attempting uninstall: PyAudio
    Found existing installation: PyAudio 0.2.14
    Uninstalling PyAudio-0.2.14:
      Successfully uninstalled PyAudio-0.2.14


In [9]:
!pip install pyaudio faster-whisper numpy

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached ctranslate2-4.8.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached av-18.0.0-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (4.5 kB)
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
Using cached av-18.0.0-cp311-abi3-manylinux_2_28_x86_64.whl (35.5 MB)
Using cached ctranslate2-4.8.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (39.5 MB)
Using cached onnxruntime-1.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (18.7 MB)


In [15]:
import pyaudio
import numpy as np
import queue
from faster_whisper import WhisperModel

# ==========================================
# 1. 오디오 및 큐(Queue) 설정
# ==========================================
RATE = 16000        # Whisper 모델이 요구하는 기본 샘플링 레이트 (16kHz)
CHUNK = 16000       # 1초 단위로 끊어서 처리 (16000 프레임 = 1초)
FORMAT = pyaudio.paInt16
CHANNELS = 1

# I/O 병목을 막기 위한 스레드 간 데이터 버퍼(Queue)
audio_queue = queue.Queue()

# ==========================================
# 2. PyAudio 콜백 함수 (생산자 역할)
# ==========================================
# 마이크에서 소리가 들어올 때마다 백그라운드 스레드에서 자동으로 실행됨
def audio_callback(in_data, frame_count, time_info, status):
    # 캡처된 바이트 데이터를 즉시 큐에 집어넣음 (I/O 블로킹 방지)
    audio_queue.put(in_data)
    return (None, pyaudio.paContinue)

# ==========================================
# 3. 모델 초기화
# ==========================================
print("모델을 로드하는 중입니다... (최초 실행 시 다운로드)")
# GPU(CUDA)를 사용할 수 있다면 'cuda', 아니면 'cpu' 사용
# compute_type="float16" (GPU) 또는 "int8" (CPU)로 최적화
# model = WhisperModel("small", device="cuda", compute_type="float16")
model = WhisperModel("small", device="cpu", compute_type="int8")
print("모델 로드 완료!")

# ==========================================


모델을 로드하는 중입니다... (최초 실행 시 다운로드)
모델 로드 완료!


이제 `PyAudio` 인스턴스를 생성하고 오디오 스트림을 열어 음성 녹음을 시작해 보겠습니다. 이 셀을 실행하면 마이크를 통해 음성 입력이 시작됩니다.

In [12]:
# ==========================================
# 4. PyAudio 스트림 시작 (소비자 역할) - 녹음 시작
# ==========================================
# Colab 환경에서는 마이크와 같은 물리적인 오디오 장치에 직접 접근할 수 없어
# PyAudio를 통한 실시간 녹음이 불가능합니다. 이 코드는 Colab에서 동작하지 않습니다.
# 대신, 미리 녹음된 오디오 파일을 업로드하여 처리하는 방식을 사용해야 합니다.
# p = pyaudio.PyAudio()

# stream = p.open(format=FORMAT,
#                 channels=CHANNELS,
#                 rate=RATE,
#                 input=True,
#                 frames_per_buffer=CHUNK,
#                 stream_callback=audio_callback) # 콜백 함수 연결

# print("스트림 시작 중...")
# stream.start_stream()
# print("스트림 시작 완료! 마이크 입력 대기 중...")

OSError: [Errno -9996] Invalid input device (no default output device)

마이크 실시간 입력이 어려우므로, 대신 오디오 파일을 업로드하여 처리하는 방법을 사용해 보겠습니다. 아래 코드를 실행하면 파일 업로드 대화 상자가 나타납니다.

In [14]:
from google.colab import files

# 파일을 업로드합니다.
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving test.mp4 to test.mp4
User uploaded file "test.mp4" with length 167672472 bytes


업로드된 비디오 파일 `test.mp4`에서 오디오 트랙을 추출해야 `faster-whisper` 모델이 처리할 수 있습니다. `ffmpeg`를 사용하여 오디오를 WAV 형식으로 추출하겠습니다.

In [16]:
# 비디오 파일에서 오디오 추출
video_file = next(iter(uploaded.keys()))
audio_file = video_file.replace('.mp4', '.wav') # 또는 다른 오디오 형식

# ffmpeg를 사용하여 오디오 추출 (Colab 환경에 기본 설치되어 있음)
!ffmpeg -i "{video_file}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{audio_file}"

print(f"오디오가 '{audio_file}'로 추출되었습니다.")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

오디오 파일이 추출되었으니, 이제 `faster-whisper` 모델을 사용하여 음성을 텍스트로 변환하겠습니다. 모델은 이미 로드되어 있습니다. 변환된 결과는 자막 형식으로 표시하고 저장할 수 있습니다.

In [17]:
import datetime

# 오디오 파일 전사(Transcribe)
print("오디오 전사를 시작합니다...")
segments, info = model.transcribe(audio_file, beam_size=5)
print(f"발견된 언어: {info.language} (확률: {info.language_probability:.2f})")

all_segments = []
print("\n--- 실시간 자막 출력 (시뮬레이션) ---")
for segment in segments:
    start_time = str(datetime.timedelta(seconds=int(segment.start)))
    end_time = str(datetime.timedelta(seconds=int(segment.end)))
    print(f"[{start_time} --> {end_time}] {segment.text}")
    all_segments.append(segment)
print("------------------------------------")
print("오디오 전사 및 실시간 자막 출력 완료!")

오디오 전사를 시작합니다...
발견된 언어: ko (확률: 1.00)

--- 실시간 자막 출력 (시뮬레이션) ---
[0:00:00 --> 0:00:04]  오이에킹 가면 가장 당황스러운 부분 중에 하나가 어떤 시스템이 있는지 담당자도 몰라
[0:00:04 --> 0:00:06]  우리가 찾아서 알려주면
[0:00:06 --> 0:00:08]  어 뭐야 이거 이거 어디 썼어요?
[0:00:08 --> 0:00:10]  관리가 안되고 방치되는 서머들
[0:00:10 --> 0:00:13]  어쨌든 회사들이 밀린 숙제들이 있는거야
[0:00:13 --> 0:00:14]  밀린 구몬들이 있어
[0:00:14 --> 0:00:17]  국정원에서 이걸 낸 이유가 뭐냐면
[0:00:17 --> 0:00:19]  너네 AI 쓰고 싶지
[0:00:19 --> 0:00:21]  가서 밀린 구몬 해와
[0:00:21 --> 0:00:23]  가서 빨리 구몬 다 해와
[0:00:27 --> 0:00:29]  회사 다니시는 분들은 아시겠지만
[0:00:29 --> 0:00:31]  사무실 컴퓨터에서는요
[0:00:31 --> 0:00:33]  인터넷이 안됩니다
[0:00:33 --> 0:00:34]  망블리 때문입니다
[0:00:34 --> 0:00:35]  카카오토
[0:00:35 --> 0:00:37]  유튜브 못하게 일부러 지금
[0:00:37 --> 0:00:38]  라고 생각하겠지만
[0:00:38 --> 0:00:39]  그게 아닙니다
[0:00:39 --> 0:00:42]  그래서 이 망블리에 대해서 한번 이야기를 해볼건데
[0:00:42 --> 0:00:44]  이 망블리 이야기의 시작이
[0:00:44 --> 0:00:47]  2000년 초부터 시작이 됩니다
[0:00:47 --> 0:00:49]  2000년 초는 전세계적으로
[0:00:49 --> 0:00:51]  해킹이 난무하던 시대였구요
[0:00:51 --> 0:00:54]  어나니머스가 뜨기 시작했던 것

KeyboardInterrupt: 

전사된 내용을 SRT 자막 파일 형식으로 저장하는 코드를 제공합니다. SRT 파일은 비디오 플레이어에서 자막을 표시하는 데 널리 사용됩니다.

In [18]:
# SRT 파일로 저장
srt_file_name = audio_file.replace('.wav', '.srt')

with open(srt_file_name, "w", encoding="utf-8") as f:
    for i, segment in enumerate(all_segments):
        start_time_str = str(datetime.timedelta(seconds=segment.start)).split('.')[0]
        end_time_str = str(datetime.timedelta(seconds=segment.end)).split('.')[0]

        # SRT 형식에 맞게 밀리초 추가
        start_ms = int((segment.start - int(segment.start)) * 1000)
        end_ms = int((segment.end - int(segment.end)) * 1000)

        f.write(f"{i + 1}\n")
        f.write(f"{start_time_str},{start_ms:03d} --> {end_time_str},{end_ms:03d}\n")
        f.write(f"{segment.text.strip()}\n\n")

print(f"자막이 '{srt_file_name}' 파일로 성공적으로 저장되었습니다.")
print("이제 이 파일을 다운로드하여 비디오와 함께 사용하거나 다른 텍스트 편집기에서 확인할 수 있습니다.")

# 저장된 SRT 파일 다운로드
files.download(srt_file_name)

자막이 'test.srt' 파일로 성공적으로 저장되었습니다.
이제 이 파일을 다운로드하여 비디오와 함께 사용하거나 다른 텍스트 편집기에서 확인할 수 있습니다.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

파일 업로드가 완료되면, 업로드된 파일 이름을 사용하여 `faster-whisper` 모델로 음성을 텍스트로 변환하는 코드를 작성할 수 있습니다.

위 셀을 실행하면 마이크가 활성화되고 음성 입력이 `audio_queue`에 지속적으로 저장됩니다. 이제 녹음된 오디오 데이터를 처리할 수 있는 코드를 추가해야 합니다. 예를 들어, 일정 시간 동안 녹음하거나 특정 트리거에 따라 녹음을 중지하고 텍스트로 변환하는 등의 작업을 할 수 있습니다.